In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('All_Data.csv', sep=';')

# list_of_feats_Causal_Discovery = ['MD 210',
#  'DB 70DBD 4 ΘΕΡΜ ΚΟΛΕΚΤΕΡ ΨΥΓΕΙΩΝ (ΝΕΡΟ ΑΠΟ ΔΕΞΑΜΕΝΗ)',
#  'DB 30DBW 20 ramposition',
#  'DB 20DBW 174 EXTRACTION STEP',
#  'DB 10DBW 14 BACKWARD PRESS',
#  'DB 20DBD 292',
#  'Setpoint position exhaust damper',
#  'DB 400DBD 34 z2 energy',
#  'T 174',
#  'MD 284 C ana kw',
#  'MD 70 sinolo ypog-mpanioy', 'Timestamp', 'Energy']

# df = df[list_of_feats_Causal_Discovery]
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df[:-120] #cutoff used in the training pipeline as well
C_D_features = ['Timestamp', 'MD 210',
 'DB 70DBD 4 ΘΕΡΜ ΚΟΛΕΚΤΕΡ ΨΥΓΕΙΩΝ (ΝΕΡΟ ΑΠΟ ΔΕΞΑΜΕΝΗ)',
 'DB 30DBW 20 ramposition',
 'DB 20DBW 174 EXTRACTION STEP',
 'DB 10DBW 14 BACKWARD PRESS',
 'DB 20DBD 292',
 'Setpoint position exhaust damper',
 'DB 400DBD 34 z2 energy',
 'T 174',
 'MD 284 C ana kw',
 'MD 70 sinolo ypog-mpanioy', 'Energy'] # features after the causal discovery analysis

df = df[C_D_features]
df

,Timestamp,MD 210,DB 70DBD 4 ΘΕΡΜ ΚΟΛΕΚΤΕΡ ΨΥΓΕΙΩΝ (ΝΕΡΟ ΑΠΟ ΔΕΞΑΜΕΝΗ),DB 30DBW 20 ramposition,DB 20DBW 174 EXTRACTION STEP,DB 10DBW 14 BACKWARD PRESS,DB 20DBD 292,Setpoint position exhaust damper,DB 400DBD 34 z2 energy,T 174,MD 284 C ana kw,MD 70 sinolo ypog-mpanioy,Energy
0,2025-02-28 13:00:00,350.000000,31.257333,1223.400000,0.533333,5.800000,327680.0,100.000000,16774.16,1032.666667,483.72,2736.063333,101.671
1,2025-02-28 13:15:00,496.666667,28.298667,1215.466667,0.866667,6.266667,275251.2,100.000000,19248.49,1024.666667,305.84,2731.106667,153.854
2,2025-02-28 13:30:00,146.666667,27.054000,803.866667,0.866667,9.600000,275251.2,98.666667,24453.65,1036.000000,364.94,2726.779333,134.813
3,2025-02-28 13:45:00,0.000000,28.497333,1031.133333,0.000000,9.133333,314572.8,99.000000,19891.34,1040.666667,368.80,2726.767333,145.956
4,2025-02-28 14:00:00,293.333333,27.597333,954.800000,1.800000,6.600000,157286.4,100.000000,22821.02,1031.333333,230.77,2725.181333,140.812
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2166,2025-03-23 02:30:00,0.000000,20.442000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.607333,14.997
2167,2025-03-23 02:45:00,0.000000,20.436000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.610000,5.174
2168,2025-03-23 03:00:00,0.000000,20.340000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.237333,5.886
2169,2025-03-23 03:15:00,0.000000,20.288000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.213333,12.474


In [3]:
 # Categorical features
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)
df['dow'] = df.index.day_name().str[:3]     # e.g. "Mon","Tue",…,"Sun"
df['hour'] = df.index.hour # 0–23
sun_low = df['dow'] == 'Sun' # basically all day on sunday
mon_low = (df['dow'] == 'Mon') & df['hour'].isin([0,1,2,3,4,5,6,7,8,9]) # basically from 0 to 9 on monday
wed_low = (df['dow'] == 'Wed') & df['hour'].isin([6,7,8,9,10,11,12,13,14]) # basically from 6 to 14 for maintenance on wednesday
sat_low = (df['dow'] == 'Sat') & df['hour'].isin([19,20,21,22,23]) # basically from 19 to 23 on saturday
df['is_low_usage'] = sun_low | mon_low | wed_low | sat_low
df.drop(columns=['dow', 'hour'], inplace=True)
df.reset_index(inplace=True)
df['is_low_usage'] = df['is_low_usage'].astype(int)
df['is_low_usage_next'] = df['is_low_usage'].shift(-1)




# # cyclical encode timestamp as extra features
# df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
# df.set_index('Timestamp', inplace=True)
# df['dow'] = df.index.day_name().str[:3]     # e.g. "Mon","Tue",…,"Sun"
# df['hour'] = df.index.hour # 0–23
# sun_low = df['dow'] == 'Sun' # basically all day on sunday
# mon_low = (df['dow'] == 'Mon') & df['hour'].isin([0,1,2,3,4,5,6,7,8,9]) # basically from 0 to 9 on monday
# wed_low = (df['dow'] == 'Wed') & df['hour'].isin([6,7,8,9,10,11,12,13,14]) # basically from 6 to 14 for maintenance on wednesday
# sat_low = (df['dow'] == 'Sat') & df['hour'].isin([19,20,21,22,23]) # basically from 19 to 23 on saturday
# df['is_low_usage'] = sun_low | mon_low | wed_low | sat_low
# df.drop(columns=['dow', 'hour'], inplace=True)
# df.reset_index(inplace=True)
# df['is_low_usage'] = df['is_low_usage'].astype(int)



# Hour of day (0–23)
df['hour_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.hour / 24)

# # Minute of hour (0–59)
# df['minute_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.minute / 60)
# df['minute_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.minute / 60)

# Compute number of days in each month
df['days_in_month'] = df['Timestamp'].dt.days_in_month
df['day_of_week_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
# Safe cyclic encoding
# df['day_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.day / df['days_in_month'])
# df['day_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.day / df['days_in_month'])
# Categorical features

# df['month_sin'] = np.sin(2 * np.pi * df['Timestamp'].dt.month / 12)
# df['month_cos'] = np.cos(2 * np.pi * df['Timestamp'].dt.month / 12)
df.drop(columns=['Timestamp', 'days_in_month'], inplace=True)
df.to_csv('energy_data.csv', index=False, sep=',')
df

,MD 210,DB 70DBD 4 ΘΕΡΜ ΚΟΛΕΚΤΕΡ ΨΥΓΕΙΩΝ (ΝΕΡΟ ΑΠΟ ΔΕΞΑΜΕΝΗ),DB 30DBW 20 ramposition,DB 20DBW 174 EXTRACTION STEP,DB 10DBW 14 BACKWARD PRESS,DB 20DBD 292,Setpoint position exhaust damper,DB 400DBD 34 z2 energy,T 174,MD 284 C ana kw,MD 70 sinolo ypog-mpanioy,Energy,is_low_usage,is_low_usage_next,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos
0,350.000000,31.257333,1223.400000,0.533333,5.800000,327680.0,100.000000,16774.16,1032.666667,483.72,2736.063333,101.671,0,0.0,-0.258819,-0.965926,-0.433884,-0.900969
1,496.666667,28.298667,1215.466667,0.866667,6.266667,275251.2,100.000000,19248.49,1024.666667,305.84,2731.106667,153.854,0,0.0,-0.258819,-0.965926,-0.433884,-0.900969
2,146.666667,27.054000,803.866667,0.866667,9.600000,275251.2,98.666667,24453.65,1036.000000,364.94,2726.779333,134.813,0,0.0,-0.258819,-0.965926,-0.433884,-0.900969
3,0.000000,28.497333,1031.133333,0.000000,9.133333,314572.8,99.000000,19891.34,1040.666667,368.80,2726.767333,145.956,0,0.0,-0.258819,-0.965926,-0.433884,-0.900969
4,293.333333,27.597333,954.800000,1.800000,6.600000,157286.4,100.000000,22821.02,1031.333333,230.77,2725.181333,140.812,0,0.0,-0.500000,-0.866025,-0.433884,-0.900969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2166,0.000000,20.442000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.607333,14.997,1,1.0,0.500000,0.866025,-0.781831,0.623490
2167,0.000000,20.436000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.610000,5.174,1,1.0,0.500000,0.866025,-0.781831,0.623490
2168,0.000000,20.340000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.237333,5.886,1,1.0,0.707107,0.707107,-0.781831,0.623490
2169,0.000000,20.288000,1.000000,0.000000,0.000000,0.0,50.000000,0.00,1000.000000,96.00,2783.213333,12.474,1,1.0,0.707107,0.707107,-0.781831,0.623490


In [4]:
import pandas as pd

df = pd.read_csv('energy_data.csv')
column_to_move = 'Energy'
df = df[[col for col in df.columns if col != column_to_move] + [column_to_move]]
df.to_csv('energy_data.csv', index=False)

In [5]:
len(df.columns)

18

In [6]:
import pandas as pd

df = pd.read_csv('energy_data.csv')
n = len(df)
train_end = int(n * 0.85)
X_train = df.iloc[:train_end]
X_train.to_csv('energy_data.csv', index=False)

In [7]:
len(X_train.columns)

18

In [8]:
X_train

,MD 210,DB 70DBD 4 ΘΕΡΜ ΚΟΛΕΚΤΕΡ ΨΥΓΕΙΩΝ (ΝΕΡΟ ΑΠΟ ΔΕΞΑΜΕΝΗ),DB 30DBW 20 ramposition,DB 20DBW 174 EXTRACTION STEP,DB 10DBW 14 BACKWARD PRESS,DB 20DBD 292,Setpoint position exhaust damper,DB 400DBD 34 z2 energy,T 174,MD 284 C ana kw,MD 70 sinolo ypog-mpanioy,is_low_usage,is_low_usage_next,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,Energy
0,350.000000,31.257333,1223.400000,0.533333,5.800000,327680.000000,100.000000,16774.16,1032.666667,483.72,2736.063333,0,0.0,-0.258819,-9.659258e-01,-0.433884,-0.900969,101.671
1,496.666667,28.298667,1215.466667,0.866667,6.266667,275251.200000,100.000000,19248.49,1024.666667,305.84,2731.106667,0,0.0,-0.258819,-9.659258e-01,-0.433884,-0.900969,153.854
2,146.666667,27.054000,803.866667,0.866667,9.600000,275251.200000,98.666667,24453.65,1036.000000,364.94,2726.779333,0,0.0,-0.258819,-9.659258e-01,-0.433884,-0.900969,134.813
3,0.000000,28.497333,1031.133333,0.000000,9.133333,314572.800000,99.000000,19891.34,1040.666667,368.80,2726.767333,0,0.0,-0.258819,-9.659258e-01,-0.433884,-0.900969,145.956
4,293.333333,27.597333,954.800000,1.800000,6.600000,157286.400000,100.000000,22821.02,1031.333333,230.77,2725.181333,0,0.0,-0.500000,-8.660254e-01,-0.433884,-0.900969,140.812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1840,26.666667,18.952000,928.733333,0.666667,11.666667,200977.066667,100.000000,36458.31,1033.333333,214.66,2735.759333,0,0.0,-0.965926,-2.588190e-01,0.974928,-0.222521,165.597
1841,254.666667,19.434667,863.800000,0.266667,3.400000,139810.133333,100.000000,31744.40,1055.333333,379.32,2733.662000,0,0.0,-0.965926,-2.588190e-01,0.974928,-0.222521,179.747
1842,0.000000,20.401333,1167.400000,1.000000,6.933333,117964.800000,100.000000,38113.84,1034.666667,239.44,2731.782000,0,0.0,-0.965926,-2.588190e-01,0.974928,-0.222521,163.784
1843,26.666667,26.781333,1150.333333,1.933333,4.733333,209715.200000,100.000000,25640.58,1035.333333,339.83,2728.936667,0,0.0,-0.965926,-2.588190e-01,0.974928,-0.222521,118.536
